In [ ]:
---原始存档
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.loan_time IS NOT NULL THEN o.loan_amt END )                        AS 放款金额
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.loan_time IS NOT NULL THEN o.fee_rate * o.loan_amt END )           AS T0放款金额_定价
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND o.loan_time IS NOT NULL THEN o.loan_amt END )            AS T7放款金额
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 AND o.loan_time IS NOT NULL THEN o.loan_amt END )           AS T30放款金额
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 AND o.loan_time IS NOT NULL THEN o.loan_amt END )                       AS 累积放款金额
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 AND o.loan_time IS NOT NULL THEN o.fee_rate * o.loan_amt END )          AS 累积放款金额_定价
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.loan_time IS NOT NULL THEN o.loan_amt * o.period END )             AS 放款金额_period

       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.risk_status = 'pass' THEN shouxin.user_no END )          AS 风控通过人
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND o.risk_status = 'pass' THEN shouxin.user_no END ) AS T7风控通过人
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 AND o.risk_status = 'pass' THEN shouxin.user_no END ) AS T30风控通过人
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 AND o.risk_status = 'pass' THEN shouxin.user_no END )         AS 累积风控通过人

       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.risk_status = 'pass' THEN o.order_amt END )                        AS T0风险通过_提现金额 ----T7风险通过_提现金额 
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND o.risk_status = 'pass' THEN o.order_amt END )            AS T7风险通过_提现金额
       
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.loan_time IS NOT NULL THEN shouxin.user_no END )         AS 放款人
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND o.loan_time IS NOT NULL THEN shouxin.user_no END ) AS T7放款人
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 AND o.loan_time IS NOT NULL THEN shouxin.user_no END ) AS T30放款人
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 AND o.loan_time IS NOT NULL THEN shouxin.user_no END )        AS 累积放款人
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.loan_time IS NOT NULL THEN shouxin.init_credit_line / 100 END )    AS 授信额度_放款
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND o.loan_time IS NOT NULL THEN shouxin.init_credit_line / 100 END ) AS 授信额度_放款7
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 AND o.loan_time IS NOT NULL THEN shouxin.init_credit_line / 100 END ) AS 授信额度_放款30
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 AND o.loan_time IS NOT NULL THEN shouxin.init_credit_line / 100 END )   AS 授信额度_放款累积
       ,COUNT(DISTINCT CASE WHEN back.user_no IS NOT NULL THEN shouxin.user_no END)                                                                   AS back_cnt
       ,COUNT(DISTINCT CASE WHEN core.md_zh = '提现页曝光' THEN shouxin.user_no END)                                                                       AS 提现页_cnt
       ,COUNT(DISTINCT CASE WHEN core.md_zh = '点击借款按钮' THEN shouxin.user_no END)                                                                      AS 点击借款按钮_cnt
       ,COUNT(DISTINCT CASE WHEN core.md_zh = '点击借款按钮' AND DATEDIFF(o.first_order_time,shouxin.created_time) = 0 THEN shouxin.user_no ELSE NULL END ) AS create_cnt_dj

In [ ]:
DROP TABLE IF EXISTS app_shouxin_jxge_lss;
CREATE TABLE app_shouxin_jxge_lss AS 

INSERT OVERWRITE TABLE app_shouxin_jxge_lss
-- 完件人群 
WITH wanjian_userno AS
(
	SELECT  shenqing.*
	FROM
	(
		SELECT  *
		FROM xyf_dwd.dwd_preloan_credit_apply_df
		WHERE pt = '${bizdate}'
		AND date(created_time) >= '2024-01-01'
		AND app IN ('xyf01')
		AND inner_app IN ('xyf01', 'xyf01_hrui02', 'xyf01_xcjr', 'xyf01_hrui01', 'xyf01_alyxy', 'xyf01_alygd', 'xyf01_alyfz', 'xyf01_zyxj01', 'xyf01_zyxjwld01', 'xyf01_zyxjzl01', 'xyf01_elm')
		AND NVL(app_activation_type, '') <> 'loan_recredit_activation' 
              QUALIFY ROW_NUMBER() OVER (PARTITION BY user_no, date(created_time) ORDER BY created_time DESC ) = 1 
	) shenqing 
       LEFT ANTI JOIN
	(
		SELECT  DISTINCT biz_flow_number
		FROM xyf_dwd.dwd_inloan_t_decision_result_detail_df
		WHERE enginecode = 'jcl_20240722000003'
		AND pt = MAX_PT('xyf_dwd.dwd_inloan_t_decision_result_detail_df')
		AND inner_app <> 'xyf01_test1' 
	) b
	ON shenqing.biz_flow_number = b.biz_flow_number
)

SELECT  TO_DATE(wanjian.created_time)                                                                                                         AS dt
       ,wanjian.created_time
       ,zhou.week_range                                                                                                                       AS wt
       ,SUBSTR(wanjian.created_time,1,7)                                                                                                      AS mt
       ,CASE WHEN 0 <= MAX(shouxin.init_credit_line / 100) AND MAX(shouxin.init_credit_line / 100) <= 500 THEN '[0-500]'
             WHEN 500 < MAX(shouxin.init_credit_line / 100) AND MAX(shouxin.init_credit_line / 100) <= 1000 THEN '(500-1000]'
             WHEN 1000 < MAX(shouxin.init_credit_line / 100) AND MAX(shouxin.init_credit_line / 100) <= 1500 THEN '(1000-1500]'
             WHEN 1500 < MAX(shouxin.init_credit_line / 100) AND MAX(shouxin.init_credit_line / 100) < 2000 THEN '(1500-2000]'
             WHEN 2000 < MAX(shouxin.init_credit_line / 100) AND MAX(shouxin.init_credit_line / 100) <= 3000 THEN '(2000-3000]'
             WHEN 3000 < MAX(shouxin.init_credit_line / 100) AND MAX(shouxin.init_credit_line / 100) <= 5000 THEN '(3000-5000]'
             WHEN 5000 < MAX(shouxin.init_credit_line / 100) AND MAX(shouxin.init_credit_line / 100) <= 10000 THEN '(5000-10000]'
             WHEN 10000 < MAX(shouxin.init_credit_line / 100) AND MAX(shouxin.init_credit_line / 100) <= 20000 THEN '(10000-20000]'
             WHEN 20000 < MAX(shouxin.init_credit_line / 100) AND MAX(shouxin.init_credit_line / 100) < 50000 THEN '(20000-50000]'
             WHEN 50000 < MAX(shouxin.init_credit_line / 100) AND MAX(shouxin.init_credit_line / 100) <= 100000 THEN '(50000-100000]'
             WHEN 100000 < MAX(shouxin.init_credit_line / 100) THEN '100000+'  ELSE '' END                               AS 授信金额_level 
       ,CASE
             WHEN shouxin.init_credit_line / 100 < 1000 THEN '49元'
             WHEN shouxin.init_credit_line / 100 < 1500 THEN '99元'
             WHEN shouxin.init_credit_line / 100 < 2000 THEN '149元'
             WHEN shouxin.init_credit_line / 100 >= 2000 AND RANDOMV3('fxvip_price_test',shouxin.user_no,2) BETWEEN 0 AND 19 THEN '149元'
             WHEN shouxin.init_credit_line / 100 >= 2000 AND RANDOMV3('fxvip_price_test',shouxin.user_no,2) BETWEEN 20 AND 99 THEN '199元' END AS 会员卡价格档位
       ,wanjian.user_no
       ,is_API半流程
       ,is_虚假给额
       ,COUNT(DISTINCT shouxin.user_no)                                                                                                       AS cnt
       ,MAX(shouxin.init_credit_line / 100)                                                                                                   AS 授信额度_shu
       ,COUNT(DISTINCT CASE WHEN xujia_type.是否api虚假给额 = 1 THEN shouxin.user_no END)                                                           AS API虚假给额
       ,COUNT(DISTINCT CASE WHEN shouxin.is_虚假给额 = '虚假给额' AND is_API半流程 = 'API半流程' THEN shouxin.user_no END)                            AS 半流程虚假给额
       
       --提现
       ,COUNT(DISTINCT CASE WHEN COALESCE(DATEDIFF(o.first_order_time,shouxin.created_time,'mi'),9999) BETWEEN 0 AND 30 THEN shouxin.user_no END ) AS 30min提现人
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 THEN shouxin.user_no END)                                   AS T0提现人
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 THEN shouxin.user_no END )                      AS T7提现人
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 1 AND 7 THEN shouxin.user_no END )                      AS T1_7提现人
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 THEN shouxin.user_no END )                     AS T30提现人
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 THEN shouxin.user_no END)                                  AS 累积提现人

       ,MAX( CASE WHEN COALESCE(DATEDIFF(o.first_order_time,shouxin.created_time,'mi'),9999) BETWEEN 0 AND 30 THEN o.order_amt END )               AS 30min提现金额
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 THEN o.order_amt END)                                                 AS T0提现金额
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 THEN o.order_amt END )                                    AS T7提现金额
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 THEN o.order_amt END )                                   AS T30提现金额
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 THEN o.order_amt END)                                                AS 累积提现金额
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 THEN o.order_amt * o.period END )                                     AS 提现金额_period

      --风控通过 
       ,COUNT(DISTINCT CASE WHEN COALESCE(DATEDIFF(o.first_order_time,shouxin.created_time,'mi'),9999) BETWEEN 0 AND 30 AND o.risk_status = 'pass' THEN shouxin.user_no END ) AS 30min风控通过人
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.risk_status = 'pass' THEN shouxin.user_no END )       AS T0风控通过人
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND o.risk_status = 'pass' THEN shouxin.user_no END ) AS T7风控通过人
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 1 AND 7 AND o.risk_status = 'pass' THEN shouxin.user_no END ) AS T1_T7风控通过人
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 AND o.risk_status = 'pass' THEN shouxin.user_no END ) AS T30风控通过人
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 AND o.risk_status = 'pass' THEN shouxin.user_no END )      AS 累积风控通过人 
      --资产 
       ,MAX( CASE WHEN COALESCE(DATEDIFF(o.first_order_time,shouxin.created_time,'mi'),9999) BETWEEN 0 AND 30 AND o.risk_status = 'pass' THEN o.order_amt END ) AS 30min风险通过_提现金额
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.risk_status = 'pass' THEN o.order_amt END )                     AS T0风险通过_提现金额
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND o.risk_status = 'pass' THEN o.order_amt END )         AS T7风险通过_提现金额 

       --放款 
       ,COUNT(DISTINCT CASE WHEN COALESCE(DATEDIFF(o.first_order_time,shouxin.created_time,'mi'),9999) BETWEEN 0 AND 30 AND o.loan_time IS NOT NULL THEN shouxin.user_no END ) AS 30min放款人
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.loan_time IS NOT NULL THEN shouxin.user_no END )      AS T0放款人
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND o.loan_time IS NOT NULL THEN shouxin.user_no END ) AS T7放款人
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 1 AND 7 AND o.loan_time IS NOT NULL THEN shouxin.user_no END ) AS T1_T7放款人
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 AND o.loan_time IS NOT NULL THEN shouxin.user_no END ) AS T30放款人
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 AND o.loan_time IS NOT NULL THEN shouxin.user_no END )     AS 累积放款人

       ,MAX( CASE WHEN COALESCE(DATEDIFF(o.first_order_time,shouxin.created_time,'mi'),9999) BETWEEN 0 AND 30 AND o.loan_time IS NOT NULL THEN o.loan_amt END ) AS 30min放款金额
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.loan_time IS NOT NULL THEN o.loan_amt END )                     AS T0放款金额
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND o.loan_time IS NOT NULL THEN o.loan_amt END )         AS T7放款金额
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 1 AND 7 AND o.loan_time IS NOT NULL THEN o.loan_amt END )         AS T1_T7放款金额
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 AND o.loan_time IS NOT NULL THEN o.loan_amt END )        AS T30放款金额
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 AND o.loan_time IS NOT NULL THEN o.loan_amt END )                    AS 累积放款金额 

       --30min会员卡 
       ,COUNT(DISTINCT CASE WHEN COALESCE(DATEDIFF(o.first_order_time,shouxin.created_time,'mi'),9999) BETWEEN 0 AND 30 AND vip.order_time IS NOT NULL THEN shouxin.user_no END ) AS 30min会员卡签约
       ,COUNT(DISTINCT CASE WHEN COALESCE(DATEDIFF(o.first_order_time,shouxin.created_time,'mi'),9999) BETWEEN 0 AND 30 AND vip.status = 3 THEN shouxin.user_no END ) AS 30min会员卡扣款
       ,MAX( CASE WHEN COALESCE(DATEDIFF(o.first_order_time,shouxin.created_time,'mi'),9999) BETWEEN 0 AND 30 AND vip.status = 3 THEN vip.real_card_price END ) AS 30min会员卡扣款金额 
       ---T0会员卡 
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND vip.order_time IS NOT NULL THEN shouxin.user_no END )   AS T0会员卡签约
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND vip.status = 3 THEN shouxin.user_no END )               AS T0会员卡扣款
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND vip.status = 3 THEN vip.real_card_price END )                     AS T0会员卡扣款金额 
       ---T7会员卡 
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND vip.order_time IS NOT NULL THEN shouxin.user_no END ) AS T7会员卡签约
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(vip.order_time,shouxin.created_time) BETWEEN 1 AND 7 AND vip.order_time IS NOT NULL THEN shouxin.user_no END ) AS T1_T7会员卡签约
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND vip.status = 3 THEN shouxin.user_no END )   AS T7会员卡扣款
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 1 AND 7 AND vip.status = 3 THEN shouxin.user_no END )   AS T1_T7会员卡扣款
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND vip.status = 3 THEN vip.real_card_price END )         AS T7会员卡扣款金额
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 1 AND 7 AND vip.status = 3 THEN vip.real_card_price END )         AS T1_T7会员卡扣款金额 
       ---T30 
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 AND vip.order_time IS NOT NULL THEN shouxin.user_no END ) AS T30会员卡签约
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 AND vip.status = 3 THEN shouxin.user_no END )  AS T30会员卡扣款
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 AND vip.status = 3 THEN vip.real_card_price END )        AS T30会员卡扣款金额 
       ---累计 
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 AND vip.order_time IS NOT NULL THEN shouxin.user_no END )  AS 累积会员卡签约
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 AND vip.status = 3 THEN shouxin.user_no END )              AS 累积会员卡扣款
       ,MAX( CASE WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 AND vip.status = 3 THEN vip.real_card_price END )                    AS 累积会员卡扣款金额
FROM 
(
	SELECT  w.user_no
	       ,w.cust_no
	       ,w.created_time
	       ,w.biz_flow_number
	       ,CASE WHEN w.inner_app IN ('xyf01') THEN 
                     CASE WHEN w.client_code IN ('MPP001000068') THEN '微信小程序'
	                   WHEN w.client_code IN ('MPP002000069') THEN '抖音小程序'  ELSE 'APP全流程' END  
               ELSE 'API半流程' END AS is_API半流程
	FROM wanjian_userno w
) wanjian
LEFT JOIN
(
	SELECT  shouxin.*
	       ,CASE WHEN xujia.biz_flow_number IS NOT NULL AND inner_app IN ('xyf01','fxk') THEN '虚假给额'
	             WHEN inner_app NOT IN ('xyf01','fxk') AND init_credit_line / 100 < 1000 THEN '虚假给额' -----半流程的虚假给额规则。此时为APP和半流程的虚假给额 
	             ELSE '非虚假给额' END AS is_虚假给额
	FROM
	(
		SELECT  *
		FROM wanjian_userno
		WHERE status = 2 --成功 
 
	) shouxin
	LEFT JOIN
	( -- 虚假给额的授信成功用户口径，biz_flow_number关联授信表 
		SELECT  biz_flow_number -- 授信biz_flow_number 
		FROM xyf_dwd.dwd_inloan_t_decision_result_detail_df
		WHERE pt = MAX_PT('xyf_dwd.dwd_inloan_t_decision_result_detail_df')
		AND enginecode = 'jcl_20240923000003'
		AND GET_JSON_OBJECT(context, "$.app_new_risk_mark_output") RLIKE 'fake_activation'
		AND decision_time >= '2024-10-15 00:00:00' 
		-- 存在少量异常数据，同一个biz_flow_number+enginename 存在多条记录；下面排序做个兜底的清洗 
              QUALIFY ROW_NUMBER() OVER (PARTITION BY biz_flow_number, date(decision_time) ORDER BY decision_time DESC ) = 1 
	) xujia
	ON shouxin.biz_flow_number = xujia.biz_flow_number
) shouxin
ON wanjian.user_no = shouxin.user_no AND DATEDIFF(wanjian.created_time, shouxin.created_time) = 0
-- 风险app人行评级
LEFT JOIN
(
	SELECT  授信_biz_flow_number
	       ,是否api虚假给额
	       ,CAST(NVL(人行评级,-1.0) AS DOUBLE) 人行评级
	FROM xyf_bi.aji_app_rank_activation_forbi 
       QUALIFY ROW_NUMBER() OVER (PARTITION BY 授信_biz_flow_number ORDER BY created_time ) = 1 ---不知道需不需要去重，但这样保险一点 
) xujia_type
ON xujia_type.授信_biz_flow_number = shouxin.biz_flow_number
-- API人行评级
LEFT JOIN
(
	SELECT  biz_flow_number                                     AS 授信_biz_flow_number
	       ,CAST(NVL(personalloan_sd_line_amtlevel_rh_api,-1.0) AS DOUBLE) 人行评级
	FROM xyf_fengkong.api_activation_base
) api_rk
ON api_rk.授信_biz_flow_number = shouxin.biz_flow_number
LEFT JOIN
(
	SELECT  day_id_iso
	       ,CONCAT(day_week01_xf_new,'至',day_weekend_xf_new) AS week_range
	FROM xyf_dim.dim_pub_date
) zhou
ON TO_DATE(wanjian.created_time) = zhou.day_id_iso
-- 问一下back是什么事件
LEFT JOIN
(
	SELECT  user_no
	       ,date(tracking_timestamp) dt
	       ,MAX(tracking_timestamp) event_time
	FROM xyf_dwd.dwd_event_tracking_log_di 
	WHERE DATE(TO_DATE(pt, "yyyyMMdd")) >= DATE_SUB(CURRENT_TIMESTAMP(), 180)
	GROUP BY  user_no
	         ,dt
) back
ON back.user_no = shouxin.user_no AND SUBSTR(back.dt, 1, 10) = SUBSTR(shouxin.created_time, 1, 10) AND shouxin.created_time < back.event_time
-- 提现页曝光、点击借款按钮埋点
LEFT JOIN
(
	SELECT  user_no
	       ,date(tracking_timestamp) dt
	       ,md_zh
	       ,MIN(tracking_timestamp) event_time
	FROM xyf_dwd.dwd_event_tracking_log_core_di
	WHERE DATE(TO_DATE(pt, "yyyyMMdd")) >= DATE_SUB(CURRENT_TIMESTAMP(), 180)
	AND md_zh IN ('提现页曝光', '点击借款按钮')
	GROUP BY  user_no
	         ,dt
	         ,md_zh
) core
ON core.user_no = shouxin.user_no AND SUBSTR(core.dt, 1, 10) = SUBSTR(shouxin.created_time, 1, 10)
-- 关联订单表
LEFT JOIN
(
	SELECT  *
	FROM xyf_dws.dws_inloan_user_order_df
	WHERE pt = '${bizdate}'
	AND app IN ('xyf01', 'fxk')
	AND business_line IN ('APP', '小程序端') -- APP首贷订单(发起维度) 
	-- AND loan_status = 'success'
	AND loan_flag = '首贷' 
) o
ON o.cust_no = shouxin.cust_no

LEFT JOIN
( --是否在提现页买卡？ 
      SELECT   loan_order_number                                                   
	       ,order_status                                                   AS  status 
	       ,real_card_price                                               
	       ,order_time
	       ,CASE WHEN order_from LIKE '%lend_before%' THEN '借款页签约'
	             WHEN order_from LIKE '%lend_after%' THEN '卡单页签约'  ELSE '其他' END AS is_借款页签约
	       ,app_user_id
	FROM xyf_dwd.dwd_inloan_vip_order_df
	WHERE pt = MAX_PT('xyf_dwd.dwd_inloan_vip_order_df') --新飞享会员卡表
	AND vip_order_number = first_vip_order_number --不看续约 
	AND order_time IS NOT NULL
	AND SUBSTR(order_time, 1, 10) >= '2024-08-01' 

	UNION ALL

	SELECT  loan_order_number                                                
	       ,CASE  WHEN order_status = 'pay_success' THEN 3  ELSE 0 END                AS status            
	       ,real_card_price                                                
	       ,order_time
	       ,CASE WHEN order_from LIKE '%lend_before%' THEN '借款页签约'
	             WHEN order_from LIKE '%lend_after%' THEN '卡单页签约'  ELSE '其他' END AS is_借款页签约
	       ,app_user_id
	FROM xyf_dwd.dwd_inloan_leap_vip_order_hf
	WHERE pt = MAX_PT('xyf_dwd.dwd_inloan_leap_vip_order_hf')
	AND vip_order_number = first_vip_order_number --不看续约 
	AND order_time IS NOT NULL
	AND SUBSTR(order_time, 1, 10) >= '2025-08-11' 
) vip 
ON vip.loan_order_number = o.first_order_number  
-- ON vip.app_user_id = o.user_no AND date(vip.order_time) = date(o.first_order_time) --大额拆单，导致飞享的关联逻辑有误，暂时改为模糊匹配
GROUP BY  TO_DATE(wanjian.created_time)
         ,wanjian.created_time
         ,wt
         ,mt
         ,wanjian.user_no
         ,is_API半流程
         ,is_虚假给额
         ,CASE
             WHEN shouxin.init_credit_line / 100 < 1000 THEN '49元'
             WHEN shouxin.init_credit_line / 100 < 1500 THEN '99元'
             WHEN shouxin.init_credit_line / 100 < 2000 THEN '149元'
             WHEN shouxin.init_credit_line / 100 >= 2000 AND RANDOMV3('fxvip_price_test',shouxin.user_no,2) BETWEEN 0 AND 19 THEN '149元'
             WHEN shouxin.init_credit_line / 100 >= 2000 AND RANDOMV3('fxvip_price_test',shouxin.user_no,2) BETWEEN 20 AND 99 THEN '199元' END 
;



SELECT  a.wt
       ,a.mt
       ,a.授信额度_shu
       ,a.is_API半流程
       ,a.is_虚假给额
       ,会员卡价格档位
       ,30min提现人                                                                AS is_30min发起
       ,30min会员卡签约                                                            AS is_30min会员卡签约
       ,30min风控通过人                                                            AS is_30min风控通过
       ,T0提现人                                                                   AS is_T0提现
       ,T0风控通过人                                                               AS is_T0风控通过
       ,T0会员卡签约                                                               AS is_T0会员卡签约
       ,CASE WHEN T1_7提现人 = 1 AND T1_T7会员卡签约 = 1 THEN 'T1-T7签约'  ELSE '未签约' END AS is_T1_T7会员卡签约
       ,CASE WHEN T0风控通过人 = 0 AND T0提现人 = 1 THEN 'T0风控拒绝' END                   AS is_reject
       ,SUM(cnt)                                                                AS cnt
       ,SUM(30min提现人)                                                           AS 30min提现人
       ,SUM(T0提现人)                                                              AS T0提现人
       ,SUM(T7提现人)                                                              AS T7提现人
       ,SUM(T1_7提现人)                                                            AS T1_7提现人
       ,SUM(T30提现人)                                                             AS T30提现人
       ,SUM(30min提现金额)                                                          AS 30min提现金额
       ,SUM(T0提现金额)                                                             AS T0提现金额
       ,SUM(T7提现金额)                                                             AS T7提现金额
       ,SUM(T30提现金额)                                                            AS T30提现金额
       ,SUM(累积提现金额)                                                             AS 累积提现金额
       ,SUM(提现金额_period)                                                        AS 提现金额_period
       ,SUM(30min风险通过_提现金额)                                                     AS 30min风险通过_提现金额
       ,SUM(T0风险通过_提现金额)                                                        AS T0风险通过_提现金额
       ,SUM(T7风险通过_提现金额)                                                        AS T7风险通过_提现金额
       ,SUM(30min放款金额)                                                          AS 30min放款金额
       ,SUM(T0放款金额)                                                             AS T0放款金额
       ,SUM(T7放款金额)                                                             AS T7放款金额
       ,SUM(T1_T7放款金额)                                                          AS T1_T7放款金额
       ,SUM(T30放款金额)                                                            AS T30放款金额
       ,SUM(30min风控通过人)                                                         AS 30min风控通过人
       ,SUM(T0风控通过人)                                                            AS T0风控通过人
       ,SUM(T7风控通过人)                                                            AS T7风控通过人
       ,SUM(CASE WHEN T1_7提现人 <> 0 THEN T1_T7风控通过人 ELSE 0 END)                  AS T1_T7风控通过人
       ,SUM(T30风控通过人)                                                           AS T30风控通过人
       ,SUM(累积风控通过人)                                                            AS 累积风控通过人
       ,SUM(30min放款人)                                                           AS 30min放款人
       ,SUM(T0放款人)                                                              AS T0放款人
       ,SUM(t7放款人)                                                              AS t7放款人
       ,SUM(CASE WHEN T1_7提现人 <> 0 THEN T1_T7放款人 ELSE 0 END)                    AS T1_T7放款人
       ,SUM(t30放款人)                                                             AS t30放款人
       ,SUM(30min会员卡签约)                                                         AS 30min会员卡签约
       ,SUM(30min会员卡扣款)                                                         AS 30min会员卡扣款
       ,SUM(30min会员卡扣款金额)                                                       AS 30min会员卡扣款金额
       ,SUM(T0会员卡签约)                                                            AS T0会员卡签约
       ,SUM(T0会员卡扣款)                                                            AS T0会员卡扣款
       ,SUM(T0会员卡扣款金额)                                                          AS T0会员卡扣款金额
       ,SUM(T7会员卡签约)                                                            AS T7会员卡签约
       ,SUM(T1_T7会员卡签约)                                                         AS T1_T7会员卡签约
       ,SUM(T7会员卡扣款)                                                            AS T7会员卡扣款
       ,SUM(T7会员卡扣款金额)                                                          AS T7会员卡扣款金额
       ,SUM(T30会员卡签约)                                                           AS T30会员卡签约
       ,SUM(T30会员卡扣款)                                                           AS T30会员卡扣款
       ,SUM(T30会员卡扣款金额)                                                         AS T30会员卡扣款金额
       ,SUM(累积会员卡签约)                                                            AS 累积会员卡签约
       ,SUM(累积会员卡扣款)                                                            AS 累积会员卡扣款
       ,SUM(累积会员卡扣款金额)                                                          AS 累积会员卡扣款金额
       ,SUM(累积放款金额)                                                             AS 累积放款金额
       ,SUM(累积提现人)                                                              AS 累积提现人
       ,SUM(累积放款人)                                                              AS 累积放款人
       ,SUM(is_T7放款且签约)                                                         AS T7放款且签约
       ,SUM(is_T1_T7放款且签约)                                                      AS is_T1T7放款且签约
       ,SUM(T1_T7会员卡扣款)                                                         AS is_T1T7扣款
       ,SUM(T1_T7会员卡扣款金额)                                                       AS T1_T7会员卡扣款金额
FROM
(
	SELECT  *
	       ,CASE WHEN T7放款人 = 1 AND T7会员卡签约 = 1 THEN 1  ELSE 0 END                        AS is_T7放款且签约
	       ,CASE WHEN T1_7提现人 <> 0 AND T1_T7放款人 = 1 AND T1_T7会员卡签约 = 1 THEN 1  ELSE 0 END AS is_T1_T7放款且签约
	FROM app_shouxin_jxge_lss
) a
GROUP BY  a.wt
         ,a.mt
         ,a.授信额度_shu
         ,a.is_API半流程
         ,a.is_虚假给额
         ,30min会员卡签约
         ,30min风控通过人
         ,30min提现人
         ,T0提现人
         ,T0风控通过人
         ,T0会员卡签约
         ,会员卡价格档位
         ,CASE WHEN T0风控通过人 = 0 AND T0提现人 = 1 THEN 'T0风控拒绝' END
         ,CASE WHEN T1_7提现人 = 1 AND T1_T7会员卡签约 = 1 THEN 'T1-T7签约'  ELSE '未签约' END



In [1]:

query='''
SELECT  a.mt
       ,a.授信金额档位
       ,a.is_API半流程
       ,a.is_虚假给额
       ,会员卡价格档位
       ,30min提现人                                                                AS is_30min发起
       ,30min会员卡签约                                                            AS is_30min会员卡签约
       ,30min风控通过人                                                            AS is_30min风控通过
       ,T0提现人                                                                   AS is_T0提现
       ,T0风控通过人                                                               AS is_T0风控通过
       ,T0会员卡签约                                                               AS is_T0会员卡签约
       ,CASE WHEN T1_7提现人 = 1 AND T1_T7会员卡签约 = 1 THEN 'T1-T7签约'  ELSE '未签约' END AS is_T1_T7会员卡签约
       ,CASE WHEN T0风控通过人 = 0 AND T0提现人 = 1 THEN 'T0风控拒绝' END                   AS is_reject
       ,SUM(cnt)                                                                AS cnt
       ,SUM(30min提现人)                                                           AS 30min提现人
       ,SUM(T0提现人)                                                              AS T0提现人
       ,SUM(T7提现人)                                                              AS T7提现人
       ,SUM(T1_7提现人)                                                            AS T1_7提现人
       ,SUM(T30提现人)                                                             AS T30提现人
       ,SUM(30min提现金额)                                                          AS 30min提现金额
       ,SUM(T0提现金额)                                                             AS T0提现金额
       ,SUM(T7提现金额)                                                             AS T7提现金额
       ,SUM(T30提现金额)                                                            AS T30提现金额
       ,SUM(累积提现金额)                                                             AS 累积提现金额
       ,SUM(提现金额_period)                                                        AS 提现金额_period
       ,SUM(30min风险通过_提现金额)                                                     AS 30min风险通过_提现金额
       ,SUM(T0风险通过_提现金额)                                                        AS T0风险通过_提现金额
       ,SUM(T7风险通过_提现金额)                                                        AS T7风险通过_提现金额
       ,SUM(30min放款金额)                                                          AS 30min放款金额
       ,SUM(T0放款金额)                                                             AS T0放款金额
       ,SUM(T7放款金额)                                                             AS T7放款金额
       ,SUM(T1_T7放款金额)                                                          AS T1_T7放款金额
       ,SUM(T30放款金额)                                                            AS T30放款金额
       ,SUM(30min风控通过人)                                                         AS 30min风控通过人
       ,SUM(T0风控通过人)                                                            AS T0风控通过人
       ,SUM(T7风控通过人)                                                            AS T7风控通过人
       ,SUM(CASE WHEN T1_7提现人 <> 0 THEN T1_T7风控通过人 ELSE 0 END)                  AS T1_T7风控通过人
       ,SUM(T30风控通过人)                                                           AS T30风控通过人
       ,SUM(累积风控通过人)                                                            AS 累积风控通过人
       ,SUM(30min放款人)                                                           AS 30min放款人
       ,SUM(T0放款人)                                                              AS T0放款人
       ,SUM(t7放款人)                                                              AS t7放款人
       ,SUM(CASE WHEN T1_7提现人 <> 0 THEN T1_T7放款人 ELSE 0 END)                    AS T1_T7放款人
       ,SUM(t30放款人)                                                             AS t30放款人
       ,SUM(30min会员卡签约)                                                         AS 30min会员卡签约
       ,SUM(30min会员卡扣款)                                                         AS 30min会员卡扣款
       ,SUM(30min会员卡扣款金额)                                                       AS 30min会员卡扣款金额
       ,SUM(T0会员卡签约)                                                            AS T0会员卡签约
       ,SUM(T0会员卡扣款)                                                            AS T0会员卡扣款
       ,SUM(T0会员卡扣款金额)                                                          AS T0会员卡扣款金额
       ,SUM(T7会员卡签约)                                                            AS T7会员卡签约
       ,SUM(T1_T7会员卡签约)                                                         AS T1_T7会员卡签约
       ,SUM(T7会员卡扣款)                                                            AS T7会员卡扣款
       ,SUM(T7会员卡扣款金额)                                                          AS T7会员卡扣款金额
       ,SUM(T30会员卡签约)                                                           AS T30会员卡签约
       ,SUM(T30会员卡扣款)                                                           AS T30会员卡扣款
       ,SUM(T30会员卡扣款金额)                                                         AS T30会员卡扣款金额
       ,SUM(累积会员卡签约)                                                            AS 累积会员卡签约
       ,SUM(累积会员卡扣款)                                                            AS 累积会员卡扣款
       ,SUM(累积会员卡扣款金额)                                                          AS 累积会员卡扣款金额
       ,SUM(累积放款金额)                                                             AS 累积放款金额
       ,SUM(累积提现人)                                                              AS 累积提现人
       ,SUM(累积放款人)                                                              AS 累积放款人
       ,SUM(is_T7放款且签约)                                                         AS T7放款且签约
       ,SUM(is_T1_T7放款且签约)                                                      AS is_T1T7放款且签约
       ,SUM(T1_T7会员卡扣款)                                                         AS is_T1T7扣款
       ,SUM(T1_T7会员卡扣款金额)                                                       AS T1_T7会员卡扣款金额
FROM
(
	SELECT  *
           ,CASE WHEN 授信额度_shu <= 500 THEN '[0-500]'
             WHEN 授信额度_shu <= 1000 THEN '(500-1000]'
             WHEN 授信额度_shu <= 1500 THEN '(1000-1500]'
             WHEN 授信额度_shu <= 2000 THEN '(1500-2000]'
             WHEN 授信额度_shu <= 3000 THEN '(2000-3000]'
             WHEN 授信额度_shu <= 5000 THEN '(3000-5000]'
             WHEN 授信额度_shu <= 10000 THEN '(5000-10000]'
             WHEN 授信额度_shu <= 20000 THEN '(10000-20000]'
             WHEN 授信额度_shu <= 50000 THEN '(20000-50000]'
             WHEN 授信额度_shu <= 100000 THEN '(50000-100000]'
             WHEN 授信额度_shu > 100000 THEN '100000+'  ELSE '' END                                 AS 授信金额档位 
	       ,CASE WHEN T7放款人 = 1 AND T7会员卡签约 = 1 THEN 1  ELSE 0 END                        AS is_T7放款且签约
	       ,CASE WHEN T1_7提现人 <> 0 AND T1_T7放款人 = 1 AND T1_T7会员卡签约 = 1 THEN 1  ELSE 0 END AS is_T1_T7放款且签约
	FROM app_shouxin_jxge_lss
    WHERE 授信额度_shu IS NOT NULL
    AND mt >= '2025-01'
) a
GROUP BY  a.mt
         ,a.授信金额档位
         ,a.is_API半流程
         ,a.is_虚假给额
         ,30min会员卡签约
         ,30min风控通过人
         ,30min提现人
         ,T0提现人
         ,T0风控通过人
         ,T0会员卡签约
         ,会员卡价格档位
         ,CASE WHEN T0风控通过人 = 0 AND T0提现人 = 1 THEN 'T0风控拒绝' END
         ,CASE WHEN T1_7提现人 = 1 AND T1_T7会员卡签约 = 1 THEN 'T1-T7签约'  ELSE '未签约' END

'''

In [2]:
import query_analysis_tool as qat
# import importlib
# importlib.reload(qat)  # 强制重新加载最新的包代码
# qat.show_odps_config()
data_stats = qat.run_query(query)

str_cols = [
    'mt', '授信金额档位', 'is_API半流程', 'is_虚假给额', '会员卡价格档位', 
    'is_t1_t7会员卡签约', 'is_reject'
]

int_cols = [
    'is_30min发起', 'is_30min会员卡签约', 'is_30min风控通过', 'is_t0提现', 'is_t0风控通过', 'is_t0会员卡签约',
    'cnt', '30min提现人', 't0提现人', 't7提现人', 't1_7提现人', 't30提现人', 
    '30min风控通过人', 't0风控通过人', 't7风控通过人', 't1_t7风控通过人', 't30风控通过人', '累积风控通过人',
    '30min放款人', 't0放款人', 't7放款人', 't1_t7放款人', 't30放款人', 
    '30min会员卡签约', '30min会员卡扣款', 't0会员卡签约', 't0会员卡扣款', 
    't7会员卡签约', 't1_t7会员卡签约', 't7会员卡扣款', 't30会员卡签约', 't30会员卡扣款', 
    '累积会员卡签约', '累积会员卡扣款', '累积提现人', '累积放款人', 't7放款且签约', 'is_t1t7放款且签约', 'is_t1t7扣款'
]

float_cols = [
    '30min提现金额', 't0提现金额', 't7提现金额', 't30提现金额', '累积提现金额', '提现金额_period',
    '30min风险通过_提现金额', 't0风险通过_提现金额', 't7风险通过_提现金额', 
    '30min放款金额', 't0放款金额', 't7放款金额', 't1_t7放款金额', 't30放款金额', '累积放款金额',
    '30min会员卡扣款金额', 't0会员卡扣款金额', 't7会员卡扣款金额', 't1_t7会员卡扣款金额', 't30会员卡扣款金额', '累积会员卡扣款金额'
]

data_stats = qat.format_dataframe_columns(
    data_stats,
    str_cols=str_cols,
    date_cols=[], 
    int_cols=int_cols,
    float_cols=float_cols
)

qat.write_dataframe_to_excel(
    file_path=r"D:\9.极限给额专题分析\分析new\极限给额专项分析0513.xlsx",
    dataframes_dict={"授信数据": data_stats},
    start_row=1,
    include_header=True
)

正在获取数据，首段 SQL: 
SELECT  a.mt
       ,a.授信金额档位
       ,a.is_API半流程 ...
成功写入工作表: 授信数据
文件已保存: D:\9.极限给额专题分析\分析new\极限给额专项分析0513.xlsx


In [ ]:
qat.clear_pivot_calculated_fields(
    file_path=r"D:\9.极限给额专题分析\新客分析\极限给额专项分析1.xlsx",
    sheet_name="授信转化",
    pivot_name="数据透视表1",
)

In [4]:
calc_fields = { 
    't0动支发起率': {"formula": "='t0提现人'/'cnt'", "number_format": "0.00%"},
    't0动支通过率': {"formula": "='t0风控通过人'/'t0提现人'", "number_format": "0.00%"},
    't0放款率': {"formula": "='t0放款人'/'cnt'", "number_format": "0.00%"},
    't0单产': {"formula": "='t0放款金额'/'cnt'", "number_format": "0.00%"},
    't0件均': {"formula": "='t0放款金额'/'t0放款人'", "number_format": "0.00%"},
    't0会员卡签约率': {"formula": "='t0会员卡签约'/'cnt'", "number_format": "0.00%"},
    't0会员卡扣款率': {"formula": "='t0会员卡扣款'/'t0会员卡签约'", "number_format": "0.00%"},

    '30min动支发起率': {"formula": "='30min提现人'/'cnt'", "number_format": "0.00%"},
    '30min动支通过率': {"formula": "='30min风控通过人'/'30min提现人'", "number_format": "0.00%"},
    '30min放款率': {"formula": "='30min放款人'/'cnt'", "number_format": "0.00%"},
    '30min单产': {"formula": "='30min放款金额'/'cnt'", "number_format": "0.00%"},
    '30min件均': {"formula": "='30min放款金额'/'30min放款人'", "number_format": "0.00%"},
    '30min会员卡签约率': {"formula": "='30min会员卡签约'/'cnt'", "number_format": "0.00%"},
    '30min会员卡扣款率': {"formula": "='30min会员卡扣款'/'30min会员卡签约'", "number_format": "0.00%"},

    't7动支发起率': {"formula": "='t7提现人'/'cnt'", "number_format": "0.00%"},
    't7动支通过率': {"formula": "='t7风控通过人'/'t7提现人'", "number_format": "0.00%"},
    't7放款率': {"formula": "='t7放款人'/'cnt'", "number_format": "0.00%"},
    't7单产': {"formula": "='t7放款金额'/'cnt'", "number_format": "0.00%"},
    't7件均': {"formula": "='t7放款金额'/'t7放款人'", "number_format": "0.00%"},
    't7会员卡签约率': {"formula": "='t7会员卡签约'/'cnt'", "number_format": "0.00%"},
    't7会员卡扣款率': {"formula": "='t7会员卡扣款'/'t7会员卡签约'", "number_format": "0.00%"},

    't30动支发起率': {"formula": "='t30提现人'/'cnt'", "number_format": "0.00%"},
    't1-7动支发起率': {"formula": "='t1_t7提现人'/'cnt'", "number_format": "0.00%"},
    't1-7动支通过率': {"formula": "='t1_t7风控通过人'/'t1_t7提现人'", "number_format": "0.00%"},
    't1-7会员卡签约率': {"formula": "='t1_t7会员卡签约'/'cnt'", "number_format": "0.00%"},
    't7会员卡单产': {"formula": "='t7会员卡扣款金额'/'cnt'", "number_format": "0.00%"},
}

qat.add_pivot_calculated_fields(
    file_path=r"D:\9.极限给额专题分析\新客分析\极限给额专项分析1.xlsx",
    sheet_name="授信转化",
    pivot_name="数据透视表1",
    fields=calc_fields
) 

[Pivot] sheet=授信转化 pivot=数据透视表1 cache_index=6
[OK] Add CalculatedField: t0动支发起率 | ='t0提现人'/'cnt'
[OK] Add to Values: t0动支发起率 | format=0.00%
[OK] Add CalculatedField: t0动支通过率 | ='t0风控通过人'/'t0提现人'
[OK] Add to Values: t0动支通过率 | format=0.00%
[OK] Add CalculatedField: t0放款率 | ='t0放款人'/'cnt'
[OK] Add to Values: t0放款率 | format=0.00%
[OK] Add CalculatedField: t0单产 | ='t0放款金额'/'cnt'
[OK] Add to Values: t0单产 | format=0.00%
[OK] Add CalculatedField: t0件均 | ='t0放款金额'/'t0放款人'
[OK] Add to Values: t0件均 | format=0.00%
[OK] Add CalculatedField: t0会员卡签约率 | ='t0会员卡签约'/'cnt'
[OK] Add to Values: t0会员卡签约率 | format=0.00%
[OK] Add CalculatedField: t0会员卡扣款率 | ='t0会员卡扣款'/'t0会员卡签约'
[OK] Add to Values: t0会员卡扣款率 | format=0.00%
[OK] Add CalculatedField: 30min动支发起率 | ='30min提现人'/'cnt'
[OK] Add to Values: 30min动支发起率 | format=0.00%
[OK] Add CalculatedField: 30min动支通过率 | ='30min风控通过人'/'30min提现人'
[OK] Add to Values: 30min动支通过率 | format=0.00%
[OK] Add CalculatedField: 30min放款率 | ='30min放款人'/'cnt'
[OK] Add to Values: 30mi

[{'field': 't1-7动支发起率',
  'reason': 'add_to_values_failed',
  'detail': "(-2147352567, '发生意外。', (0, 'Microsoft Excel', None, None, 0, -2146827284), None)"},
 {'field': 't1-7动支通过率',
  'reason': 'add_to_values_failed',
  'detail': "(-2147352567, '发生意外。', (0, 'Microsoft Excel', None, None, 0, -2146827284), None)"}]

#### 投诉代码

In [ ]:
drop TABLE IF EXISTS xyf_bi_dev.kesu_users;

CREATE TABLE if not exists xyf_bi_dev.kesu_users AS

SELECT  *
FROM
(
	SELECT  a.*
	       ,CASE WHEN order_number_source NOT IN ('APP','API') THEN b.loan_status  ELSE c.loan_status END loanstatus
	       ,CASE WHEN order_number_source NOT IN ('APP','API') THEN DATEDIFF(cjdate,c.first_order_time)  ELSE 0 END timediff1
	       ,CASE WHEN order_number_source NOT IN ('APP','API') THEN a.order_number  ELSE c.order_number END order_number1
	       ,CASE WHEN order_number_source NOT IN ('APP','API') THEN b.first_order_time  ELSE c.first_order_time END first_order_time1
	FROM
	(
		SELECT  month_cj
		       ,is_zq
		       ,cust_no
		       ,id_card_number
		       ,MAX(task_type_name_new) task_type_name_new
		       ,MAX(order_number_source) order_number_source
		       ,MAX(channel_1_type_new) channel_1_type_new
		       ,MAX(b_score_level) b_card_level
		       ,MAX(education) education
		       ,MAX(monthly_income) monthly_income
		       ,MAX(if_loan) xinlaoke
		       ,MAX(asset_type) asset_type
		       ,MIN(date_cj) cjdate
		       ,MIN(order_number) order_number
		       ,MAX(vip_order_no_new) feixiang_number
		       ,MAX(s_vip_order_number) feiyue_number
		       ,MAX(cust_status) cust_status
		FROM xyf_ads.bi_ts_gd_wide_table_v3 a1
		WHERE pt = max_pt('xyf_ads.bi_ts_gd_wide_table_v3')
		AND month_cj >= '2025-01'
		AND cust_no is not null
		GROUP BY  month_cj
		         ,is_zq
		         ,cust_no
		         ,id_card_number
	) a
	LEFT JOIN
	(
		SELECT  order_number
		       ,loan_status
		       ,first_order_time
		FROM xyf_dws.dws_inloan_user_order_df
		WHERE pt = max_pt('xyf_dws.dws_inloan_user_order_df') 
	) b
	ON a.order_number = b.order_number
	LEFT JOIN
	(
		SELECT  cust_no
		       ,loan_status
		       ,order_number
		       ,first_order_time
		FROM xyf_dws.dws_inloan_user_order_df
		WHERE pt = max_pt('xyf_dws.dws_inloan_user_order_df') 
	) c
	ON a.cust_no = c.cust_no AND date(c.first_order_time) <= cjdate
) 
QUALIFY ROW_NUMBER() OVER ( PARTITION BY month_cj, is_zq, cust_no , cjdate, order_number_source, id_card_number ORDER BY timediff1 ASC ) = 1 




DROP TABLE IF EXISTS xyf_jingying_dev.lss_wzp_tmp;
CREATE TABLE xyf_jingying_dev.lss_wzp_tmp AS
SELECT  substring(first_order_time,1,7) mon1
       ,loan_status 放款状态
       ,xinlaoke 新老客
       ,asset_type_flag 资产类型
       ,first_amt_type 授信金额
       ,renhang_level 授信评级
       ,是否额外放开
       ,ever_feiyue 飞跃会员
       ,ever_feixiang 飞享会员
       ,b_card_score b卡分
       ,last_activation_line_amt 发标时额度
       ,fabiaozaihui 发标时是否在会
       ,month_income 月收入
       ,education 学历
       ,is_虚假给额
       ,br_3m_fy 三个月百融查征
       ,风险原始定价
       ,overdue1
       ,COUNT(order_number) cnt
       ,COUNT(case WHEN task_type_name_new = '保险咨询' THEN order_number else null end) 保险咨询
       ,COUNT(case WHEN task_type_name_new = '催收问题' THEN order_number else null end) 催收问题
       ,COUNT(case WHEN task_type_name_new = '其他' THEN order_number else null end) 其他
       ,COUNT(case WHEN task_type_name_new = '其他退款' THEN order_number else null end) 其他退款
       ,COUNT(case WHEN task_type_name_new = '减免费退款' THEN order_number else null end) 减免费退款
       ,COUNT(case WHEN task_type_name_new = '征信问题' THEN order_number else null end) 征信问题
       ,COUNT(case WHEN task_type_name_new = '放款' THEN order_number else null end) 放款
       ,COUNT(case WHEN task_type_name_new = '现金贷退款' THEN order_number else null end) 现金贷退款
       ,COUNT(case WHEN task_type_name_new = '电销问题' THEN order_number else null end) 电销问题
       ,COUNT(case WHEN task_type_name_new = '系统问题' THEN order_number else null end) 系统问题
       ,COUNT(case WHEN task_type_name_new = '证明问题' THEN order_number else null end) 证明问题
       ,COUNT(case WHEN task_type_name_new = '账务' THEN order_number else null end) 账务
       ,COUNT(case WHEN task_type_name_new = '费用问题' THEN order_number else null end) 费用问题
       ,COUNT(case WHEN task_type_name_new = '逾期费退款' THEN order_number else null end) 逾期费退款
       ,COUNT(case WHEN task_type_name_new = '黑猫系统投诉' THEN order_number else null end) 黑猫系统投诉
       ,COUNT(case WHEN task_type_name_new = '保险咨询' AND is_zq = '是' THEN order_number else null end) 保险咨询_重渠
       ,COUNT(case WHEN task_type_name_new = '催收问题' AND is_zq = '是' THEN order_number else null end) 催收问题_重渠
       ,COUNT(case WHEN task_type_name_new = '其他' AND is_zq = '是' THEN order_number else null end) 其他_重渠
       ,COUNT(case WHEN task_type_name_new = '其他退款' AND is_zq = '是' THEN order_number else null end) 其他退款_重渠
       ,COUNT(case WHEN task_type_name_new = '减免费退款' AND is_zq = '是' THEN order_number else null end) 减免费退款_重渠
       ,COUNT(case WHEN task_type_name_new = '征信问题' AND is_zq = '是' THEN order_number else null end) 征信问题_重渠
       ,COUNT(case WHEN task_type_name_new = '放款' AND is_zq = '是' THEN order_number else null end) 放款_重渠
       ,COUNT(case WHEN task_type_name_new = '现金贷退款' AND is_zq = '是' THEN order_number else null end) 现金贷退款_重渠
       ,COUNT(case WHEN task_type_name_new = '电销问题' AND is_zq = '是' THEN order_number else null end) 电销问题_重渠
       ,COUNT(case WHEN task_type_name_new = '系统问题' AND is_zq = '是' THEN order_number else null end) 系统问题_重渠
       ,COUNT(case WHEN task_type_name_new = '证明问题' AND is_zq = '是' THEN order_number else null end) 证明问题_重渠
       ,COUNT(case WHEN task_type_name_new = '账务' AND is_zq = '是' THEN order_number else null end) 账务_重渠
       ,COUNT(case WHEN task_type_name_new = '费用问题' AND is_zq = '是' THEN order_number else null end) 费用问题_重渠
       ,COUNT(case WHEN task_type_name_new = '逾期费退款' AND is_zq = '是' THEN order_number else null end) 逾期费退款_重渠
       ,COUNT(case WHEN task_type_name_new = '黑猫系统投诉' AND is_zq = '是' THEN order_number else null end) 黑猫系统投诉_重渠
       ,SUM(tousu) 投诉数量
       ,SUM(case WHEN is_zq = '是' THEN 1 else 0 end ) 重渠投诉数量
FROM
(
	SELECT  a.*
	       ,CASE WHEN br_3m_fy = 0 THEN '0'
	             WHEN br_3m_fy <= 5 THEN '1-5'
	             WHEN br_3m_fy <= 15 THEN '6-15'
	             WHEN br_3m_fy <= 25 THEN '16-25'
	             WHEN br_3m_fy > 25 THEN '>25'  ELSE 'other' END br_3m_fy
	FROM
	(
		SELECT  a.order_number
		       ,a.loan_status
		       ,a.first_order_time
		       ,a.cust_no
		       ,a.xinlaoke
		       ,a.asset_type_flag
		       ,CASE WHEN b.order_number2 is not null THEN 1  ELSE 0 END tousu
		       ,is_zq
		       ,task_type_name_new
		       ,CASE WHEN first_credit_succ_amt <= 5000 THEN '小于5000'
		             WHEN first_credit_succ_amt <= 10000 THEN '5k-1W'
		             WHEN first_credit_succ_amt <= 30000 THEN '1W-3W'
		             WHEN first_credit_succ_amt <= 50000 THEN '3W-5W'
		             WHEN first_credit_succ_amt > 50000 THEN '5W+'  ELSE 0 END first_amt_type
		       ,CASE WHEN first_withdraw_succ_rh_level < 2 THEN 1
		             WHEN first_withdraw_succ_rh_level < 3 THEN 2
		             WHEN first_withdraw_succ_rh_level < 4 THEN 3
		             WHEN first_withdraw_succ_rh_level < 5 THEN 4
		             WHEN first_withdraw_succ_rh_level >= 5 THEN 5  ELSE 0 END renhang_level
		       ,是否额外放开
		       ,CASE WHEN vip_leap.cust_no is not null THEN 1  ELSE 0 END ever_feiyue
		       ,CASE WHEN vip.cust_no is not null THEN 1  ELSE 0 END ever_feixiang
		       ,CASE WHEN b_card_model < 3 THEN 'AB'
		             WHEN b_card_model < 5 THEN 'CD'
		             WHEN b_card_model < 7 THEN 'EF'
		             WHEN b_card_model >= 7 THEN 'G+'  ELSE 'other' END b_card_score
		       ,CASE WHEN last_activation_line_amt <= 5000 THEN '小于5k'
		             WHEN last_activation_line_amt <= 10000 THEN '5k-1W'
		             WHEN last_activation_line_amt <= 30000 THEN '1W-3W'
		             WHEN last_activation_line_amt <= 50000 THEN '3W-5W'
		             WHEN last_activation_line_amt > 50000 THEN '5W+'  ELSE 'other' END last_activation_line_amt
		       ,fabiaozaihui
		       ,month_income
		       ,CASE WHEN d.education = '高中/中专 /技校' THEN '高中/中专/技校'
		             WHEN d.education is null THEN '未知'  ELSE d.education END education
		       ,is_虚假给额
		       ,CASE WHEN overdue = 0 THEN '0'
		             WHEN overdue <= 3 THEN '0-3'
		             WHEN overdue > 3 THEN '3+'  ELSE 'other' END overdue1
		       ,CASE WHEN op.ori_risk_price = 0.24 AND op2.ori_order_number IS NULL THEN 0.24  ELSE 0.36 END AS 风险原始定价
		FROM
		(
			SELECT  a.order_number
			       ,loan_status
			       ,first_order_time
			       ,first_order_number
			       ,cust_no
			       ,CASE WHEN a2.order_number is not null THEN 'API转APP'
			             WHEN business_line IN ('APP','小程序端') AND loan_flag = '首贷' THEN 'APP首贷'
			             WHEN business_line IN ('APP','小程序端') AND loan_flag IN ('复贷','加贷') THEN 'APP复贷'
			             WHEN business_line IN ('API') THEN 'API'  ELSE 'other' END xinlaoke
			       ,asset_type_flag
			FROM xyf_dws.dws_inloan_user_order_df a
			LEFT JOIN
			(
				SELECT  order_number
				FROM xyf_bi_dev.api_app_2_ord_v1120
			) a2
			ON a.first_order_number = a2.order_number
			WHERE pt = max_pt('xyf_dws.dws_inloan_user_order_df')
			AND date(first_order_time) >= '2025-01-01' 
		) a
		LEFT JOIN
		(
			SELECT  ori_order_number
			       ,ori_risk_price --, ori_risk_price_type 
			FROM xyf_dwd.dwd_inloan_loan_apply_main_df
			WHERE pt = MAX_PT('xyf_dwd.dwd_inloan_loan_apply_main_df') --7月1号之后的订单才有风险原始定价 
			AND DATE(date_created) >= '2025-07-01' 
		) op
		ON a.first_order_number = op.ori_order_number
		LEFT JOIN
		(
			SELECT  ori_order_number
			FROM xyf_jingying.fy_history_risk_price_modify --实际风险定价为36的订单记录（老客） 
		) op2
		ON a.first_order_number = op2.ori_order_number
		LEFT JOIN
		(
			SELECT  is_zq
			       ,task_type_name_new
			       ,CASE WHEN order_number not LIKE '%202%' THEN order_number1  ELSE order_number END order_number2
			FROM xyf_bi_dev.kesu_users
		) b
		ON a.order_number = b.order_number2
		LEFT JOIN
		(
			SELECT  cust_no
			       ,first_credit_succ_amt
			       ,first_withdraw_succ_rh_level
			FROM xyf_ads.ads_feature_custno_app_customer_df
			WHERE pt = max_pt("xyf_ads.ads_feature_custno_app_customer_df")
			AND product_type = 'personal_loan'
			AND app IN ('fxk', 'xyf01') 
			QUALIFY ROW_NUMBER() OVER (PARTITION BY cust_no ORDER BY first_credit_succ_time DESC) = 1 
		) c
		ON a.cust_no = c.cust_no
		LEFT JOIN xyf_bi.wzq_order_table d
		ON a.order_number = d.order_number
		LEFT JOIN
		(
			SELECT  cust_no
			       ,MIN(order_time) feiyue_time
			FROM xyf_dwd.dwd_inloan_leap_vip_order_hf
			WHERE pt = MAX_PT('xyf_dwd.dwd_inloan_leap_vip_order_hf')
			AND date(order_time) >= '2025-05-24'
			GROUP BY  cust_no
		) vip_leap
		ON a.cust_no = vip_leap.cust_no AND feiyue_time <= first_order_time
		LEFT JOIN
		(
			SELECT  cust_no
			       ,MIN(first_order_time) feixiang_time
			FROM xyf_dwd.dwd_user_vip_order_df
			WHERE pt = MAX_PT('xyf_dwd.dwd_user_vip_order_df')
			GROUP BY  cust_no
		) vip
		ON a.cust_no = vip.cust_no AND feixiang_time <= first_order_time
		LEFT JOIN
		(
			SELECT  cust_no
			       ,pt
			       ,MIN(b_card_model) b_card_model
			       ,MAX(last_activation_line_amt) last_activation_line_amt
			       ,MAX(0) fabiaozaihui
			FROM xyf_ads.ads_user_market_portfolio_label_df
			WHERE pt >= '20250101'
			AND pt <= '20250602'
			GROUP BY  cust_no
			         ,pt
			UNION ALL
			SELECT  cust_no
			       ,pt
			       ,MIN(b_card_model) b_card_model
			       ,MAX(last_activation_line_amt) last_activation_line_amt
			       ,MAX(cast(vip_zaihui AS int)) fabiaozaihui
			FROM xyf_ads.ads_user_market_portfolio_oldcust_label_df
			WHERE pt >= '20250603'
			GROUP BY  cust_no
			         ,pt
		) kequnchi
		ON a.cust_no = kequnchi.cust_no AND replace(date(dateadd(to_date(a.first_order_time), -1, 'dd')), '-', '') = kequnchi.pt
		LEFT JOIN
		(
			SELECT  cust_no
			       ,MAX(case WHEN monthly_income IN ('1200以内','1200-2500') THEN '2500以内' WHEN monthly_income IN ('2500-4000','4000-6000') THEN '2500-6000' WHEN monthly_income IN ('6000-10000') THEN '6000-10000' else monthly_income end) month_income
			       ,MAX(education) education
			FROM xyf_dwd.dwd_user_portrait_info_df
			WHERE pt = MAX_PT('xyf_dwd.dwd_user_portrait_info_df')
			GROUP BY  cust_no
		) f
		ON a.cust_no = f.cust_no
		LEFT JOIN
		(
			SELECT  first_order_number
			       ,is_虚假给额
			FROM xyf_jingying_dev.lss_ewfk_first_loan_info
		) g
		ON a.first_order_number = g.first_order_number
		LEFT JOIN
		(
			SELECT  order_number
			       ,MAX(max_overdue_days) overdue
			FROM xyf_dws.dws_repay_user_order_df
			WHERE pt = max_pt('xyf_dws.dws_repay_user_order_df')
			GROUP BY  order_number
		) d1
		ON a.order_number = d1.order_number
	) a
	LEFT JOIN
	(
		SELECT  cust_no
		       ,MAX(id_card_number) id_card_number
		FROM xyf_dim.dim_user_app_basic_info_df
		WHERE pt = MAX_PT('xyf_dim.dim_user_app_basic_info_df')
		GROUP BY  cust_no
	) b
	ON a.cust_no = b.cust_no
	LEFT JOIN
	(
		SELECT  id_card_number
		       ,created_time
		       ,coalesce(bairong_als_402,0) br_3m_fy
		FROM xyf_dwd.dwd_third_bairong_union_df
		WHERE pt = max_pt('xyf_dwd.dwd_third_bairong_union_df') 
	) d1
	ON b.id_card_number = d1.id_card_number AND a.first_order_time >= d1.created_time qualify ROW_NUMBER() over(PARTITION BY a.order_number ORDER BY d1.created_time DESC) = 1
)
GROUP BY  mon1
         ,loan_status
         ,xinlaoke
         ,asset_type_flag
         ,first_amt_type
         ,renhang_level
         ,是否额外放开
         ,ever_feiyue
         ,ever_feixiang
         ,b_card_score
         ,last_activation_line_amt
         ,fabiaozaihui
         ,month_income
         ,education
         ,is_虚假给额
         ,br_3m_fy
         ,风险原始定价
         ,overdue1

In [5]:

query='''


SELECT  substring(first_order_time,1,7) mon1
       ,loan_status 放款状态
       ,xinlaoke 新老客
       ,asset_type_flag 资产类型
       ,first_amt_type 授信金额
       ,is_虚假给额
       ,COUNT(order_number) cnt
       ,COUNT(case WHEN task_type_name_new = '保险咨询' THEN order_number else null end) 保险咨询
       ,COUNT(case WHEN task_type_name_new = '催收问题' THEN order_number else null end) 催收问题
       ,COUNT(case WHEN task_type_name_new = '其他' THEN order_number else null end) 其他
       ,COUNT(case WHEN task_type_name_new = '其他退款' THEN order_number else null end) 其他退款
       ,COUNT(case WHEN task_type_name_new = '减免费退款' THEN order_number else null end) 减免费退款
       ,COUNT(case WHEN task_type_name_new = '征信问题' THEN order_number else null end) 征信问题
       ,COUNT(case WHEN task_type_name_new = '放款' THEN order_number else null end) 放款
       ,COUNT(case WHEN task_type_name_new = '现金贷退款' THEN order_number else null end) 现金贷退款
       ,COUNT(case WHEN task_type_name_new = '电销问题' THEN order_number else null end) 电销问题
       ,COUNT(case WHEN task_type_name_new = '系统问题' THEN order_number else null end) 系统问题
       ,COUNT(case WHEN task_type_name_new = '证明问题' THEN order_number else null end) 证明问题
       ,COUNT(case WHEN task_type_name_new = '账务' THEN order_number else null end) 账务
       ,COUNT(case WHEN task_type_name_new = '费用问题' THEN order_number else null end) 费用问题
       ,COUNT(case WHEN task_type_name_new = '逾期费退款' THEN order_number else null end) 逾期费退款
       ,COUNT(case WHEN task_type_name_new = '黑猫系统投诉' THEN order_number else null end) 黑猫系统投诉
       ,COUNT(case WHEN task_type_name_new = '保险咨询' AND is_zq = '是' THEN order_number else null end) 保险咨询_重渠
       ,COUNT(case WHEN task_type_name_new = '催收问题' AND is_zq = '是' THEN order_number else null end) 催收问题_重渠
       ,COUNT(case WHEN task_type_name_new = '其他' AND is_zq = '是' THEN order_number else null end) 其他_重渠
       ,COUNT(case WHEN task_type_name_new = '其他退款' AND is_zq = '是' THEN order_number else null end) 其他退款_重渠
       ,COUNT(case WHEN task_type_name_new = '减免费退款' AND is_zq = '是' THEN order_number else null end) 减免费退款_重渠
       ,COUNT(case WHEN task_type_name_new = '征信问题' AND is_zq = '是' THEN order_number else null end) 征信问题_重渠
       ,COUNT(case WHEN task_type_name_new = '放款' AND is_zq = '是' THEN order_number else null end) 放款_重渠
       ,COUNT(case WHEN task_type_name_new = '现金贷退款' AND is_zq = '是' THEN order_number else null end) 现金贷退款_重渠
       ,COUNT(case WHEN task_type_name_new = '电销问题' AND is_zq = '是' THEN order_number else null end) 电销问题_重渠
       ,COUNT(case WHEN task_type_name_new = '系统问题' AND is_zq = '是' THEN order_number else null end) 系统问题_重渠
       ,COUNT(case WHEN task_type_name_new = '证明问题' AND is_zq = '是' THEN order_number else null end) 证明问题_重渠
       ,COUNT(case WHEN task_type_name_new = '账务' AND is_zq = '是' THEN order_number else null end) 账务_重渠
       ,COUNT(case WHEN task_type_name_new = '费用问题' AND is_zq = '是' THEN order_number else null end) 费用问题_重渠
       ,COUNT(case WHEN task_type_name_new = '逾期费退款' AND is_zq = '是' THEN order_number else null end) 逾期费退款_重渠
       ,COUNT(case WHEN task_type_name_new = '黑猫系统投诉' AND is_zq = '是' THEN order_number else null end) 黑猫系统投诉_重渠
       ,SUM(tousu) 投诉数量
       ,SUM(case WHEN is_zq = '是' THEN 1 else 0 end ) 重渠投诉数量
FROM
(
	SELECT  a.*
	       ,CASE WHEN br_3m_fy = 0 THEN '0'
	             WHEN br_3m_fy <= 5 THEN '1-5'
	             WHEN br_3m_fy <= 15 THEN '6-15'
	             WHEN br_3m_fy <= 25 THEN '16-25'
	             WHEN br_3m_fy > 25 THEN '>25'  ELSE 'other' END br_3m_fy
	FROM
	(
		SELECT  a.order_number
		       ,a.loan_status
		       ,a.first_order_time
		       ,a.cust_no
		       ,a.xinlaoke
		       ,a.asset_type_flag
		       ,CASE WHEN b.order_number2 is not null THEN 1  ELSE 0 END tousu
		       ,is_zq
		       ,task_type_name_new
		       ,CASE WHEN first_credit_succ_amt <= 500 THEN '[0-500]'
		             WHEN first_credit_succ_amt <= 1000 THEN '(500-1000]'
		             WHEN first_credit_succ_amt <= 1500 THEN '(1000-1500]'
		             WHEN first_credit_succ_amt <= 2000 THEN '(1500-2000]'
		             WHEN first_credit_succ_amt <= 3000 THEN '(2000-3000]'
		             WHEN first_credit_succ_amt <= 5000 THEN '(3000-5000]'
		             WHEN first_credit_succ_amt <= 10000 THEN '(5000-10000]'
		             WHEN first_credit_succ_amt <= 20000 THEN '(10000-20000]'
		             WHEN first_credit_succ_amt <= 50000 THEN '(20000-50000]'
		             WHEN first_credit_succ_amt <= 100000 THEN '(50000-100000]'
		             WHEN first_credit_succ_amt > 100000 THEN '100000+'  ELSE '' END first_amt_type
		       ,CASE WHEN first_withdraw_succ_rh_level < 2 THEN 1
		             WHEN first_withdraw_succ_rh_level < 3 THEN 2
		             WHEN first_withdraw_succ_rh_level < 4 THEN 3
		             WHEN first_withdraw_succ_rh_level < 5 THEN 4
		             WHEN first_withdraw_succ_rh_level >= 5 THEN 5  ELSE 0 END renhang_level
		       ,是否额外放开
		       ,CASE WHEN vip_leap.cust_no is not null THEN 1  ELSE 0 END ever_feiyue
		       ,CASE WHEN vip.cust_no is not null THEN 1  ELSE 0 END ever_feixiang
		       ,month_income
		       ,CASE WHEN d.education = '高中/中专 /技校' THEN '高中/中专/技校'
		             WHEN d.education is null THEN '未知'  ELSE d.education END education
		       ,is_虚假给额
		       ,CASE WHEN overdue = 0 THEN '0'
		             WHEN overdue <= 3 THEN '0-3'
		             WHEN overdue > 3 THEN '3+'  ELSE 'other' END overdue1
		       ,CASE WHEN op.ori_risk_price = 0.24 AND op2.ori_order_number IS NULL THEN 0.24  ELSE 0.36 END AS 风险原始定价
		FROM
		(
			SELECT  a.order_number
			       ,loan_status
			       ,first_order_time
			       ,first_order_number
			       ,cust_no
			       ,CASE WHEN a2.order_number is not null THEN 'API转APP'
			             WHEN business_line IN ('APP','小程序端') AND loan_flag = '首贷' THEN 'APP首贷'
			             WHEN business_line IN ('APP','小程序端') AND loan_flag IN ('复贷','加贷') THEN 'APP复贷'
			             WHEN business_line IN ('API') THEN 'API'  ELSE 'other' END xinlaoke
			       ,asset_type_flag
			FROM xyf_dws.dws_inloan_user_order_df a
			LEFT JOIN
			(
				SELECT  order_number
				FROM xyf_bi_dev.api_app_2_ord_v1120
			) a2
			ON a.first_order_number = a2.order_number
			WHERE pt = max_pt('xyf_dws.dws_inloan_user_order_df')
			AND date(first_order_time) >= '2025-01-01' 
		) a
		LEFT JOIN
		(
			SELECT  ori_order_number
			       ,ori_risk_price --, ori_risk_price_type 
			FROM xyf_dwd.dwd_inloan_loan_apply_main_df
			WHERE pt = MAX_PT('xyf_dwd.dwd_inloan_loan_apply_main_df') --7月1号之后的订单才有风险原始定价 
			AND DATE(date_created) >= '2025-07-01' 
		) op
		ON a.first_order_number = op.ori_order_number
		LEFT JOIN
		(
			SELECT  ori_order_number
			FROM xyf_jingying.fy_history_risk_price_modify --实际风险定价为36的订单记录（老客） 
		) op2
		ON a.first_order_number = op2.ori_order_number
		LEFT JOIN
		(
			SELECT  is_zq
			       ,task_type_name_new
			       ,CASE WHEN order_number not LIKE '%202%' THEN order_number1  ELSE order_number END order_number2
			FROM xyf_bi_dev.kesu_users
		) b
		ON a.order_number = b.order_number2
		LEFT JOIN
		(
			SELECT  cust_no
			       ,first_credit_succ_amt
			       ,first_withdraw_succ_rh_level
			FROM xyf_ads.ads_feature_custno_app_customer_df
			WHERE pt = max_pt("xyf_ads.ads_feature_custno_app_customer_df")
			AND product_type = 'personal_loan'
			AND app IN ('fxk', 'xyf01') 
			QUALIFY ROW_NUMBER() OVER (PARTITION BY cust_no ORDER BY first_credit_succ_time DESC) = 1 
		) c
		ON a.cust_no = c.cust_no
		LEFT JOIN xyf_bi.wzq_order_table d
		ON a.order_number = d.order_number
		LEFT JOIN
		(
			SELECT  cust_no
			       ,MIN(order_time) feiyue_time
			FROM xyf_dwd.dwd_inloan_leap_vip_order_hf
			WHERE pt = MAX_PT('xyf_dwd.dwd_inloan_leap_vip_order_hf')
			AND date(order_time) >= '2025-05-24'
			GROUP BY  cust_no
		) vip_leap
		ON a.cust_no = vip_leap.cust_no AND feiyue_time <= first_order_time
		LEFT JOIN
		(
			SELECT  cust_no
			       ,MIN(first_order_time) feixiang_time
			FROM xyf_dwd.dwd_user_vip_order_df
			WHERE pt = MAX_PT('xyf_dwd.dwd_user_vip_order_df')
			GROUP BY  cust_no
		) vip
		ON a.cust_no = vip.cust_no AND feixiang_time <= first_order_time
		LEFT JOIN
		(
			SELECT  cust_no
			       ,MAX(case WHEN monthly_income IN ('1200以内','1200-2500') THEN '2500以内' WHEN monthly_income IN ('2500-4000','4000-6000') THEN '2500-6000' WHEN monthly_income IN ('6000-10000') THEN '6000-10000' else monthly_income end) month_income
			       ,MAX(education) education
			FROM xyf_dwd.dwd_user_portrait_info_df
			WHERE pt = MAX_PT('xyf_dwd.dwd_user_portrait_info_df')
			GROUP BY  cust_no
		) f
		ON a.cust_no = f.cust_no
		LEFT JOIN
		(
			SELECT  first_order_number
			       ,is_虚假给额
			FROM xyf_jingying_dev.lss_ewfk_first_loan_info
		) g
		ON a.first_order_number = g.first_order_number
		LEFT JOIN
		(
			SELECT  order_number
			       ,MAX(max_overdue_days) overdue
			FROM xyf_dws.dws_repay_user_order_df
			WHERE pt = max_pt('xyf_dws.dws_repay_user_order_df')
			GROUP BY  order_number
		) d1
		ON a.order_number = d1.order_number
	) a
	LEFT JOIN
	(
		SELECT  cust_no
		       ,MAX(id_card_number) id_card_number
		FROM xyf_dim.dim_user_app_basic_info_df
		WHERE pt = MAX_PT('xyf_dim.dim_user_app_basic_info_df')
		GROUP BY  cust_no
	) b
	ON a.cust_no = b.cust_no
	LEFT JOIN
	(
		SELECT  id_card_number
		       ,created_time
		       ,coalesce(bairong_als_402,0) br_3m_fy
		FROM xyf_dwd.dwd_third_bairong_union_df
		WHERE pt = max_pt('xyf_dwd.dwd_third_bairong_union_df') 
	) d1
	ON b.id_card_number = d1.id_card_number AND a.first_order_time >= d1.created_time qualify ROW_NUMBER() over(PARTITION BY a.order_number ORDER BY d1.created_time DESC) = 1
)
GROUP BY  mon1
         ,loan_status
         ,xinlaoke
         ,asset_type_flag
         ,first_amt_type
         ,is_虚假给额

'''

In [7]:
import query_analysis_tool as qat

data_stats = qat.run_query(query)

str_cols = [
    'mon1', '放款状态', '新老客', '资产类型', '授信金额', '授信评级',
    '是否额外放开', '飞跃会员', '飞享会员', '月收入', '学历',
    'is_虚假给额', '三个月百融查征', '风险原始定价', 'overdue1'
]

int_cols = [
    'cnt',
    '保险咨询', '催收问题', '其他', '其他退款', '减免费退款', '征信问题', '放款', '现金贷退款',
    '电销问题', '系统问题', '证明问题', '账务', '费用问题', '逾期费退款', '黑猫系统投诉',
    '保险咨询_重渠', '催收问题_重渠', '其他_重渠', '其他退款_重渠', '减免费退款_重渠', '征信问题_重渠',
    '放款_重渠', '现金贷退款_重渠', '电销问题_重渠', '系统问题_重渠', '证明问题_重渠',
    '账务_重渠', '费用问题_重渠', '逾期费退款_重渠', '黑猫系统投诉_重渠',
    '投诉数量', '重渠投诉数量'
]

float_cols = []

data_stats = qat.format_dataframe_columns(
    data_stats,
    str_cols=str_cols,
    date_cols=[], 
    int_cols=int_cols,
    float_cols=float_cols
)
qat.write_dataframe_to_excel(
    file_path=r"D:\9.极限给额专题分析\新客分析\极限给额专项分析.xlsx",
    dataframes_dict={"投诉数据": data_stats},
    start_row=1,
    include_header=True
)


正在获取数据，首段 SQL: 


SELECT  substring(first_order_time,1,7) mon1
   ...
成功写入工作表: 投诉数据
文件已保存: D:\9.极限给额专题分析\新客分析\极限给额专项分析.xlsx


In [8]:
calc_fields = { 
    '保险咨询率': {"formula": "='保险咨询'/'cnt'", "number_format": "0.00%"},
    '催收问题率': {"formula": "='催收问题'/'cnt'", "number_format": "0.00%"},
    '其他率': {"formula": "='其他'/'cnt'", "number_format": "0.00%"},
    '其他退款率': {"formula": "='其他退款'/'cnt'", "number_format": "0.00%"},
    '减免费退款率': {"formula": "='减免费退款'/'cnt'", "number_format": "0.00%"},
    '征信问题率': {"formula": "='征信问题'/'cnt'", "number_format": "0.00%"},
    '放款率': {"formula": "='放款'/'cnt'", "number_format": "0.00%"},
    '现金贷退款率': {"formula": "='现金贷退款'/'cnt'", "number_format": "0.00%"},
    '电销问题率': {"formula": "='电销问题'/'cnt'", "number_format": "0.00%"},
    '系统问题率': {"formula": "='系统问题'/'cnt'", "number_format": "0.00%"},
    '证明问题率': {"formula": "='证明问题'/'cnt'", "number_format": "0.00%"},
    '账务率': {"formula": "='账务'/'cnt'", "number_format": "0.00%"},
    '费用问题率': {"formula": "='费用问题'/'cnt'", "number_format": "0.00%"},
    '逾期费退款率': {"formula": "='逾期费退款'/'cnt'", "number_format": "0.00%"},
    '黑猫系统投诉率': {"formula": "='黑猫系统投诉'/'cnt'", "number_format": "0.00%"},

    '保险咨询_重渠率': {"formula": "='保险咨询_重渠'/'cnt'", "number_format": "0.00%"},
    '催收问题_重渠率': {"formula": "='催收问题_重渠'/'cnt'", "number_format": "0.00%"},
    '其他_重渠率': {"formula": "='其他_重渠'/'cnt'", "number_format": "0.00%"},
    '其他退款_重渠率': {"formula": "='其他退款_重渠'/'cnt'", "number_format": "0.00%"},
    '减免费退款_重渠率': {"formula": "='减免费退款_重渠'/'cnt'", "number_format": "0.00%"},
    '征信问题_重渠率': {"formula": "='征信问题_重渠'/'cnt'", "number_format": "0.00%"},
    '放款_重渠率': {"formula": "='放款_重渠'/'cnt'", "number_format": "0.00%"},
    '现金贷退款_重渠率': {"formula": "='现金贷退款_重渠'/'cnt'", "number_format": "0.00%"},
    '电销问题_重渠率': {"formula": "='电销问题_重渠'/'cnt'", "number_format": "0.00%"},
    '系统问题_重渠率': {"formula": "='系统问题_重渠'/'cnt'", "number_format": "0.00%"},
    '证明问题_重渠率': {"formula": "='证明问题_重渠'/'cnt'", "number_format": "0.00%"},
    '账务_重渠率': {"formula": "='账务_重渠'/'cnt'", "number_format": "0.00%"},
    '费用问题_重渠率': {"formula": "='费用问题_重渠'/'cnt'", "number_format": "0.00%"},
    '逾期费退款_重渠率': {"formula": "='逾期费退款_重渠'/'cnt'", "number_format": "0.00%"},
    '黑猫系统投诉_重渠率': {"formula": "='黑猫系统投诉_重渠'/'cnt'", "number_format": "0.00%"},

    '投诉数量率': {"formula": "='投诉数量'/'cnt'", "number_format": "0.00%"},
    '重渠投诉数量率': {"formula": "='重渠投诉数量'/'cnt'", "number_format": "0.00%"},

}

qat.add_pivot_calculated_fields(
    file_path=r"D:\9.极限给额专题分析\新客分析\极限给额专项分析.xlsx",
    sheet_name="投诉情况",
    pivot_name="数据透视表1",
    fields=calc_fields
) 

[Pivot] sheet=投诉情况 pivot=数据透视表1 cache_index=7
[OK] Add CalculatedField: 保险咨询率 | ='保险咨询'/'cnt'
[OK] Add to Values: 保险咨询率 | format=0.00%
[OK] Add CalculatedField: 催收问题率 | ='催收问题'/'cnt'
[OK] Add to Values: 催收问题率 | format=0.00%
[OK] Add CalculatedField: 其他率 | ='其他'/'cnt'
[OK] Add to Values: 其他率 | format=0.00%
[OK] Add CalculatedField: 其他退款率 | ='其他退款'/'cnt'
[OK] Add to Values: 其他退款率 | format=0.00%
[OK] Add CalculatedField: 减免费退款率 | ='减免费退款'/'cnt'
[OK] Add to Values: 减免费退款率 | format=0.00%
[OK] Add CalculatedField: 征信问题率 | ='征信问题'/'cnt'
[OK] Add to Values: 征信问题率 | format=0.00%
[OK] Add CalculatedField: 放款率 | ='放款'/'cnt'
[OK] Add to Values: 放款率 | format=0.00%
[OK] Add CalculatedField: 现金贷退款率 | ='现金贷退款'/'cnt'
[OK] Add to Values: 现金贷退款率 | format=0.00%
[OK] Add CalculatedField: 电销问题率 | ='电销问题'/'cnt'
[OK] Add to Values: 电销问题率 | format=0.00%
[OK] Add CalculatedField: 系统问题率 | ='系统问题'/'cnt'
[OK] Add to Values: 系统问题率 | format=0.00%
[OK] Add CalculatedField: 证明问题率 | ='证明问题'/'cnt'
[OK] Add to Values: 证明

[]

### 投诉状态

In [ ]:
SELECT  month_cj
       ,is_zq
       ,cust_no
       ,id_card_number
       ,task_type_name_new
       ,fund_name
       ,task_number
       ,date_cj
       ,channel_1_type_new
       ,fund_jg
       ,fund_source
       ,create_time
       ,order_number
       ,if_loan
FROM xyf_ads.bi_ts_gd_wide_table_v3 a1
WHERE pt = max_pt('xyf_ads.bi_ts_gd_wide_table_v3')
--and is_zq = '是' （这里是在判断是否是重渠）（你可以根据cust_no来匹配也可以通过order_number，但order_number可能会出现匹配不上的事情） 


In [ ]:
DROP TABLE IF EXISTS lss_wzp_tmp;
CREATE TABLE lss_wzp_tmp AS
SELECT  substring(first_order_time,1,7) mon1
       ,loan_status 放款状态
       ,xinlaoke 新老客
       ,asset_type_flag 资产类型
       ,first_amt_type 授信金额
       ,renhang_level AS 授信评级
       ,是否额外放开
       ,ever_feiyue 有过飞跃会员
       ,ever_feixiang 有过飞享会员
       ,b_card_score b卡分
       ,飞跃在会
       ,飞享在会
       ,monthly_income 月收入
       ,education 学历
       ,is_虚假给额
       ,0 三个月百融多头
       ,风险原始定价
       ,可用额度区间
       ,在贷状态
       ,降额标志
       ,提额卡标志
       ,CASE WHEN vip_flow_group = 'fy_group_1' THEN '可选流'
             WHEN vip_flow_group = 'fy_group_2' THEN '转化流'  ELSE 'other 'end vip_flow_group
       ,tijie
       ,cust_status
       ,cap_type
       ,COUNT(distinct order_number) cnt
       ,COUNT(case WHEN task_type_name_new = '催收问题' THEN order_number else null end) 催收问题
       ,COUNT(case WHEN task_type_name_new = '征信问题' THEN order_number else null end) 征信问题
       ,COUNT(case WHEN task_type_name_new = '证明问题' THEN order_number else null end) 证明问题
       ,COUNT(case WHEN task_type_name_new = '费用问题' THEN order_number else null end) 费用问题
       ,COUNT(case WHEN is_zq = '是' THEN order_number else null end) 重渠总数
       ,COUNT(case WHEN task_type_name_new = '催收问题' AND is_zq = '是' THEN order_number else null end) 催收问题_重渠
       ,COUNT(case WHEN task_type_name_new = '征信问题' AND is_zq = '是' THEN order_number else null end) 征信问题_重渠
       ,COUNT(case WHEN task_type_name_new = '证明问题' AND is_zq = '是' THEN order_number else null end) 证明问题_重渠
       ,COUNT(case WHEN task_type_name_new = '费用问题' AND is_zq = '是' THEN order_number else null end) 费用问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '费用问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 7 AND to_date(DATEADD(first_order_time,7,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,1),1,7) <= substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) fpd7费用问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '费用问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 15 AND to_date(DATEADD(first_order_time,15,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,1),1,7) <= substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) fpd15费用问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '费用问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 30 AND to_date(DATEADD(first_order_time,30,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,1),1,7) <= substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob1费用问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '费用问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 60 AND to_date(DATEADD(first_order_time,60,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,2),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob2费用问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '费用问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 90 AND to_date(DATEADD(first_order_time,90,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,3),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob3费用问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '费用问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 120 AND to_date(DATEADD(first_order_time,120,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,4),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob4费用问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '费用问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 150 AND to_date(DATEADD(first_order_time,150,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,5),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob5费用问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '费用问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 180 AND to_date(DATEADD(first_order_time,180,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,6),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob6费用问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '费用问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 210 AND to_date(DATEADD(first_order_time,210,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,7),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob7费用问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '费用问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 240 AND to_date(DATEADD(first_order_time,240,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,8),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob8费用问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '费用问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 270 AND to_date(DATEADD(first_order_time,270,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,9),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end) ,0)mob9费用问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '费用问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 300 AND to_date(DATEADD(first_order_time,300,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,10),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob10费用问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '费用问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 330 AND to_date(DATEADD(first_order_time,330,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,11),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob11费用问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '费用问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 360 AND to_date(DATEADD(first_order_time,360,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,12),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob12费用问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '催收问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 7 AND to_date(DATEADD(first_order_time,7,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,1),1,7) <= substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) fpd7催收问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '催收问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 15 AND to_date(DATEADD(first_order_time,15,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,1),1,7) <= substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) fpd15催收问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '催收问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 30 AND to_date(DATEADD(first_order_time,30,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,1),1,7) <= substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob1催收问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '催收问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 60 AND to_date(DATEADD(first_order_time,60,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,2),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob2催收问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '催收问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 90 AND to_date(DATEADD(first_order_time,90,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,3),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob3催收问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '催收问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 120 AND to_date(DATEADD(first_order_time,120,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,4),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob4催收问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '催收问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 150 AND to_date(DATEADD(first_order_time,150,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,5),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob5催收问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '催收问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 180 AND to_date(DATEADD(first_order_time,180,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,6),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob6催收问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '催收问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 210 AND to_date(DATEADD(first_order_time,210,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,7),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob7催收问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '催收问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 240 AND to_date(DATEADD(first_order_time,240,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,8),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end) ,0) mob8催收问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '催收问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 270 AND to_date(DATEADD(first_order_time,270,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,9),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob9催收问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '催收问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 300 AND to_date(DATEADD(first_order_time,300,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,10),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob10催收问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '催收问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 330 AND to_date(DATEADD(first_order_time,330,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,11),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob11催收问题_重渠
       ,nullif(COUNT(case WHEN task_type_name_new = '催收问题' AND is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 360 AND to_date(DATEADD(first_order_time,360,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,12),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob12催收问题_重渠
       ,nullif(COUNT(case WHEN is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 7 AND to_date(DATEADD(first_order_time,7,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,1),1,7) <= substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) fpd7_重渠
       ,nullif(COUNT(case WHEN is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 15 AND to_date(DATEADD(first_order_time,15,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,1),1,7) <= substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) fpd15_重渠
       ,nullif(COUNT(case WHEN is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 30 AND to_date(DATEADD(first_order_time,30,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,1),1,7) <= substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob1_重渠
       ,nullif(COUNT(case WHEN is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 60 AND to_date(DATEADD(first_order_time,60,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,2),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob2_重渠
       ,nullif(COUNT(case WHEN is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 90 AND to_date(DATEADD(first_order_time,90,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,3),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob3_重渠
       ,nullif(COUNT(case WHEN is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 120 AND to_date(DATEADD(first_order_time,120,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,4),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob4_重渠
       ,nullif(COUNT(case WHEN is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 150 AND to_date(DATEADD(first_order_time,150,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,5),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob5_重渠
       ,nullif(COUNT(case WHEN is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 180 AND to_date(DATEADD(first_order_time,180,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,6),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob6_重渠
       ,nullif(COUNT(case WHEN is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 210 AND to_date(DATEADD(first_order_time,210,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,7),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob7_重渠
       ,nullif(COUNT(case WHEN is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 240 AND to_date(DATEADD(first_order_time,240,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,8),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob8_重渠
       ,nullif(COUNT(case WHEN is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 270 AND to_date(DATEADD(first_order_time,270,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,9),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob9_重渠
       ,nullif(COUNT(case WHEN is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 300 AND to_date(DATEADD(first_order_time,300,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,10),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob10_重渠
       ,nullif(COUNT(case WHEN is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 330 AND to_date(DATEADD(first_order_time,330,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,11),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob11_重渠
       ,nullif(COUNT(case WHEN is_zq = '是' AND DATEDIFF(cjdate,first_order_time) <= 360 AND to_date(DATEADD(first_order_time,360,'dd')) < CURRENT_DATE() AND substring(add_months(first_order_time,12),1,7) < substring(CURRENT_DATE(),1,7) THEN order_number else null end),0) mob12_重渠
       ,SUM(tousu) 投诉数量
       ,SUM(case WHEN is_zq = '是' THEN 1 else 0 end ) 重渠投诉数量
       ,SUM(loan_amt) 重渠权重
       ,COUNT(distinct cust_no) cust_cnt
       ,SUM(period) 总期限
FROM
(
	SELECT  a.order_number
	       ,a.loan_status
	       ,a.first_order_time
	       ,loan_amt
	       ,a.cust_no
	       ,a.xinlaoke
	       ,month_cj
	       ,a.period， a.asset_type_flag
	       ,CASE WHEN b.order_number1 is not null THEN 1  ELSE 0 END tousu
	       ,is_zq
	       ,task_type_name_new
	       ,CASE WHEN first_credit_succ_amt <= 5000 THEN '小于5000'
	             WHEN first_credit_succ_amt <= 10000 THEN '5k-1W'
	             WHEN first_credit_succ_amt <= 30000 THEN '1W-3W'
	             WHEN first_credit_succ_amt <= 50000 THEN '3W-5W'
	             WHEN first_credit_succ_amt > 50000 THEN '5W+'  ELSE 0 END first_amt_type
	       ,CASE WHEN credit_level < 2 THEN '1'
	             WHEN credit_level < 3 THEN '2'
	             WHEN credit_level < 4 THEN '3'
	             WHEN credit_level < 5 THEN '4'
	             WHEN credit_level < 6 THEN '5'
	             WHEN credit_level < 7 THEN '6'  ELSE '7+' END renhang_level
	       ,是否额外放开
	       ,CASE WHEN vip_leap.cust_no is not null THEN 1  ELSE 0 END ever_feiyue
	       ,CASE WHEN vip.cust_no is not null THEN 1  ELSE 0 END ever_feixiang
	       ,CASE WHEN b_card_model IN ('A1','A2','A') THEN 'A'
	             WHEN b_card_model IN ('B') THEN 'B'
	             WHEN b_card_model IN ('C1','C2','D') THEN 'CD'
	             WHEN b_card_model IN ('E','F') THEN 'EF'
	             WHEN b_card_model IN ('G','H','I','J') THEN 'G+'  ELSE 'other' END b_card_score
	       ,飞跃在会
	       ,飞享在会
	       ,f.monthly_income
	       ,f.education
	       ,vip_flow_group
	       ,is_虚假给额
	       ,风险原始定价
	       ,cjdate --case WHEN br_3m_fy = 0 THEN '0' 
 --when br_3m_fy <= 5 THEN '1-5' 
 --when br_3m_fy <= 15 THEN '6-15' 
 --when br_3m_fy <= 25 THEN '16-25' 
 --when br_3m_fy > 25 THEN ' > 25' 
 -- else 'other' end br_3m_fy, 
	       ,CASE WHEN settle_period is not null THEN 1  ELSE 0 END tijie
	       ,cust_status
	       ,cap_type
	       ,可用额度区间
	       ,在贷状态
	       ,降额标志
	       ,提额卡标志
	FROM
	(
		SELECT  a.order_number2 order_number
		       ,loan_status
		       ,first_order_time
		       ,first_order_number
		       ,loan_amt
		       ,period
		       ,a.cust_no
		       ,CASE WHEN a2.order_number is not null THEN 'api2app复贷'
		             WHEN business_line IN ('APP','小程序端') AND loan_flag = '首贷' AND a3.cust_no is not null THEN 'api2app首贷'
		             WHEN business_line IN ('APP','小程序端') AND loan_flag = '首贷' THEN 'APP首贷'
		             WHEN business_line IN ('APP','小程序端') AND loan_flag IN ('复贷','加贷') THEN 'APP复贷'
		             WHEN business_line IN ('API') THEN 'API'  ELSE 'other' END xinlaoke
		       ,asset_type_flag
		       ,loan_time
		       ,user_no
		FROM
		(
			SELECT  *
			FROM xyf_dws.dws_inloan_user_order_df LATERAL VIEW explode(split(order_number, ',')) tmp AS order_number2
			WHERE pt = max_pt('xyf_dws.dws_inloan_user_order_df')
			AND date(first_order_time) >= '2025-01-01'
			AND substring(first_order_time, 1, 7) <= substring(CURRENT_DATE(), 1, 7 ) 
		) a
		LEFT JOIN
		(
			SELECT  order_number
			FROM xyf_bi_dev.api_app_2_ord_v1120
		) a2
		ON a.first_order_number = a2.order_number
		LEFT JOIN
		(
			SELECT  cust_no
			FROM xyf_dwd.dwd_preloan_credit_apply_df
			WHERE pt = MAX_PT('xyf_dwd.dwd_preloan_credit_apply_df')
			AND app IN ('xyf01', 'fxk', 'cxh')
			AND status = 2
			AND inner_app REGEXP 'xyf01_'
			AND inner_app ! REGEXP 'mzsk|fxd|cxh'
			AND inner_app NOT IN ('xyf01_hrui01', 'xyf01_hrui02', 'xyf01_alygd', 'xyf01_alyxy', 'xyf01_alyfz', 'xyf01_xcjr', 'xyf01_zyxj01', 'xyf01_zyxjwld01', 'xyf01_zyxjzl01', 'xyf01_elm', 'xyf01_alytm', 'xyf01_ssyh', 'xyf01_wld01', 'xyf01_360jt04')
			GROUP BY  cust_no
		) a3
		ON a.cust_no = a3.cust_no
		WHERE pt = max_pt('xyf_dws.dws_inloan_user_order_df')
		AND date(first_order_time) >= '2025-01-01'
		AND substring(first_order_time, 1, 7) <= substring(CURRENT_DATE(), 1, 7 ) 
	) a
	LEFT JOIN
	(
		SELECT  order_number
		       ,settle_period
		FROM xyf_dws.dws_repay_user_order_df
		WHERE pt = max_pt('xyf_dws.dws_repay_user_order_df')
		GROUP BY  order_number
		         ,settle_period
	) a1
	ON a1.order_number = a.order_number
	LEFT JOIN
	(
		SELECT  order_number1
		       ,MAX(is_zq) is_zq
		       ,MAX(task_type_name_new) task_type_name_new
		       ,MIN(cjdate) cjdate
		       ,MIN(month_cj) month_cj
		       ,MAX(cust_status) cust_status
		FROM xyf_bi_dev.kesu_users
		GROUP BY  order_number1
	) b
	ON a.order_number = b.order_number1
	LEFT JOIN
	(
		SELECT  cust_no
		       ,MIN(order_time) feiyue_time
		FROM xyf_dwd.dwd_inloan_leap_vip_order_hf
		WHERE pt = MAX_PT('xyf_dwd.dwd_inloan_leap_vip_order_hf')
		AND date(order_time) >= '2025-05-24'
		GROUP BY  cust_no
	) vip_leap
	ON a.cust_no = vip_leap.cust_no AND feiyue_time <= first_order_time
	LEFT JOIN
	(
		SELECT  cust_no
		       ,MIN(first_order_time) feixiang_time
		FROM xyf_dwd.dwd_user_vip_order_df
		WHERE pt = MAX_PT('xyf_dwd.dwd_user_vip_order_df')
		GROUP BY  cust_no
	) vip
	ON a.cust_no = vip.cust_no AND feixiang_time <= first_order_time
	LEFT JOIN
	(
		SELECT  order_number
		       ,CASE WHEN monthly_income IN ('1,200以内','1,200~2,500','1200以内','1200~2500') THEN '2500以内'
		             WHEN monthly_income IN ('2,500~4,000','2500~4000','4000~6000','4,000~6,000') THEN '2500-6000'
		             WHEN monthly_income IN ('6,000~10,000','6000-10000',' >=6,000','6000~10000','>=6,000' ) THEN '6000-10000'
		             WHEN monthly_income IN ('10000~20000','10,000~20,000') THEN '10000-20000'
		             WHEN monthly_income IN ('20000~50000','20,000~50,000','20000-50000','20000 以上') THEN '20000-50000'
		             WHEN monthly_income IN ('50000以上','50,000以上') THEN '50000以上'  ELSE '未知' END monthly_income
		       ,CASE WHEN education IN ('高中/中专／技校','高中/中专 /技校','高中/中专/技校','高中/中专技校') THEN '高中/中专/技校'
		             WHEN education = '专科' THEN '专科'
		             WHEN education = '初中及以下' THEN '初中及以下'
		             WHEN education = '硕士' THEN '硕士'
		             WHEN education = '本科' THEN '本科'
		             WHEN education = '博士及以上' THEN '博士及以上'  ELSE '未知' END education
		       ,bairong_3m_fy br_3m_fy
		       ,是否额外放开
		       ,可用额度区间
		       ,在贷状态
		       ,降额标志
		       ,当笔是否签约_tek 提额卡标志
		       ,当笔是否签约_leap 飞跃在会
		       ,当笔是否签约_fx 飞享在会
		       ,存量贷中风险等级 b_card_model
		FROM xyf_jingying.wxy_order_monitor
	) f
	ON a.order_number = f.order_number
	LEFT JOIN
	(
		SELECT  order_number
		       ,风险原始定价
		       ,vip_flow_group
		FROM xyf_jingying.weekly_analysis_report_df_lss
	) g
	ON a.order_number = g.order_number
	LEFT JOIN
	(
		SELECT  first_order_number
		       ,is_虚假给额
		FROM xyf_jingying.lss_ewfk_first_loan_info
	) g1
	ON a.first_order_number = g1.first_order_number
	LEFT JOIN
	(
		SELECT  order_number
		       ,cap_type
		FROM xyf_ads.ads_fin_clear_loan_df_02_00
		WHERE pay_month >= '2024-07'
		GROUP BY  order_number
		         ,cap_type
	) c1
	ON a.order_number = c1.order_number
	LEFT JOIN
	(
		SELECT  a.cust_no
		       ,a.init_credit_line/100 first_credit_succ_amt
		       ,CASE WHEN a.inner_app = 'xyf01' THEN rk.人行评级  ELSE api_rk.personalloan_sd_line_amtlevel_rh_api END credit_level
		FROM xyf_dwd.dwd_preloan_credit_apply_df a
		LEFT JOIN xyf_bi.aji_app_rank_activation_forbi rk
		ON rk.授信_biz_flow_number = a.biz_flow_number

		LEFT JOIN xyf_fengkong.api_full_process_userprofile_insert api_rk
		ON api_rk.biz_flow_number = a.biz_flow_number
		WHERE a.pt = MAX_PT('xyf_dwd.dwd_preloan_credit_apply_df')
		AND a.status = 2 QUALIFY ROW_NUMBER() OVER (PARTITION BY a.cust_no ORDER BY a.created_time DESC) = 1 
	) sx
	ON sx.cust_no = a.cust_no
)
GROUP BY  mon1
         ,loan_status
         ,xinlaoke
         ,asset_type_flag
         ,first_amt_type
         ,renhang_level
         ,是否额外放开
         ,有过飞跃会员
         ,有过飞享会员
         ,b卡分
         ,飞跃在会
         ,飞享在会
         ,月收入
         ,学历
         ,is_虚假给额
         ,风险原始定价
         ,vip_flow_group
         ,tijie
         ,cust_status
         ,cap_type
         ,三个月百融多头
         ,可用额度区间
         ,在贷状态
         ,降额标志
         ,提额卡标志